# Nettoyage du Texte — Préparation pour le Modèle

## Contexte
Avant d'entraîner un modèle de machine learning, le texte brut
doit être nettoyé. Les majuscules, la ponctuation et les mots
trop courants ("the", "is", "a"...) n'apportent aucune information
utile au modèle.

## Ce notebook
Dans ce notebook on va :
1. Charger les données du notebook précédent
2. Mettre le texte en minuscules
3. Supprimer la ponctuation et les chiffres
4. Supprimer les stopwords (mots inutiles)
5. Lemmatiser les mots (ramener à la forme de base)
6. Sauvegarder le texte nettoyé pour le prochain notebook

## 1. Chargement des librairies

In [1]:
import pandas as pd
import re  # pour les expressions régulières (supprimer ponctuation)
import nltk  # librairie de traitement du langage naturel

# Télécharger les ressources NLTK nécessaires
nltk.download("stopwords")   # liste des mots inutiles
nltk.download("wordnet")     # dictionnaire pour la lemmatisation

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print(" Librairies chargées avec succès !")

 Librairies chargées avec succès !


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mouwa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mouwa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 2. Chargement des données

On charge le fichier produit par le notebook précédent.

In [2]:
df = pd.read_csv("../outputs/01_data_exploration.csv")

print(f"Nombre d'avis : {len(df)}")
print(f"Colonnes : {df.columns.tolist()}")
df.head(3)

Nombre d'avis : 568454
Colonnes : ['Text', 'sentiment']


,Text,sentiment
0,I have bought several of the Vitality canned d...,positif
1,Product arrived labeled as Jumbo Salted Peanut...,negatif
2,This is a confection that has been around a fe...,positif


## 3. Nettoyage du texte

On applique 7 étapes de nettoyage dans l'ordre :
1. **Minuscules** → "AMAZING" devient "amazing"
2. **Suppression balises HTML** → "br", "div" disparaissent
3. **Expansion des contractions** → "don't" devient "do not"
4. **Suppression ponctuation/chiffres** → "!!!", "123" disparaissent
5. **Suppression stopwords** → "the", "is", "a" disparaissent
6. **Suppression mots courts** → "br", "ve", "u" disparaissent
7. **Lemmatisation** → "loved" devient "love"

In [3]:
# Initialiser le lemmatiseur
lemmatizer = WordNetLemmatizer()

# Liste des mots inutiles en anglais
stop_words = set(stopwords.words("english"))

def nettoyer_texte(texte):
    # Étape 1 : mettre en minuscules
    texte = texte.lower()

    # Étape 2 : supprimer les balises HTML
    texte = re.sub(r"<.*?>", "", texte)

    # Étape 3 : développer les contractions AVANT de supprimer la ponctuation
    contractions = {
        "i've"   : "i have",
        "i'm"    : "i am",
        "i'll"   : "i will",
        "i'd"    : "i would",
        "don't"  : "do not",
        "doesn't": "does not",
        "didn't" : "did not",
        "isn't"  : "is not",
        "wasn't" : "was not",
        "aren't" : "are not",
        "wouldn't": "would not",
        "couldn't": "could not",
        "it's"   : "it is",
        "that's" : "that is",
        "there's": "there is",
        "they're": "they are",
        "we're"  : "we are",
        "you're" : "you are",
        "they've": "they have",
        "we've"  : "we have",
        "you've" : "you have",
    }
    for contraction, expansion in contractions.items():
        texte = texte.replace(contraction, expansion)

    # Étape 4 : supprimer la ponctuation et les chiffres
    texte = re.sub(r"[^a-z\s]", "", texte)

    # Étape 5 : supprimer les stopwords
    mots = texte.split()
    mots = [mot for mot in mots if mot not in stop_words]

    # Étape 6 : supprimer les mots trop courts (1-2 caractères)
    mots = [mot for mot in mots if len(mot) > 2]

    # Étape 7 : lemmatiser chaque mot
    mots = [lemmatizer.lemmatize(mot) for mot in mots]

    return " ".join(mots)

# Tester sur un exemple avant d'appliquer sur tout le dataset
exemple = "This product is AMAZING!!! I loved it so much... Buy it NOW!!!"
print("Avant :", exemple)
print("Après :", nettoyer_texte(exemple))

Avant : This product is AMAZING!!! I loved it so much... Buy it NOW!!!
Après : product amazing loved much buy


### Résultat du test

| Avant | Après |
|---|---|
| "This product is AMAZING!!!" | "product amazing" |
| "I loved it so much..." | "loved much" |
| "Buy it NOW!!!" | "buy" |

Les 4 étapes fonctionnent correctement.
On applique maintenant le nettoyage sur les 568 454 avis.

In [4]:
# Appliquer le nettoyage sur tout le dataset
print("Nettoyage en cours...")

df["text_nettoye"] = df["Text"].apply(nettoyer_texte)

print(" Nettoyage terminé !")
print(f"\nExemple :")
print(f"Avant : {df['Text'].iloc[0][:100]}")
print(f"Après : {df['text_nettoye'].iloc[0][:100]}")

Nettoyage en cours...
 Nettoyage terminé !

Exemple :
Avant : I have bought several of the Vitality canned dog food products and have found them all to be of good
Après : bought several vitality canned dog food product found good quality product look like stew processed 


## 4. Vérification de la qualité du nettoyage

On vérifie que le nettoyage n'a pas créé de problèmes :
- Des textes vides après nettoyage ?
- La longueur moyenne des textes a-t-elle bien diminué ?

In [5]:
# Vérifier les textes vides après nettoyage
textes_vides = df["text_nettoye"].str.strip().eq("").sum()
print(f"Textes vides après nettoyage : {textes_vides}")

# Comparer la longueur moyenne avant/après
longueur_avant = df["Text"].str.split().str.len().mean()
longueur_apres = df["text_nettoye"].str.split().str.len().mean()

print(f"\nLongueur moyenne avant nettoyage : {longueur_avant:.0f} mots")
print(f"Longueur moyenne après nettoyage : {longueur_apres:.0f} mots")
print(f"Réduction : {((longueur_avant - longueur_apres) / longueur_avant * 100):.0f}%")

Textes vides après nettoyage : 8

Longueur moyenne avant nettoyage : 80 mots
Longueur moyenne après nettoyage : 39 mots
Réduction : 52%


### Résultats

- Longueur moyenne réduite de 80 à 41 mots **(−49%)**
- 1 texte vide détecté après nettoyage,  on va le supprimer

La réduction de 49% est un bon signe car on a supprimé les mots inutiles sans perdre l'information importante.

In [6]:
# Supprimer le texte vide
df = df[df["text_nettoye"].str.strip() != ""]

print(f"Nombre d'avis après suppression : {len(df)}")

Nombre d'avis après suppression : 568446


## 5. Sauvegarde des données nettoyées

On sauvegarde uniquement les colonnes utiles pour le prochain notebook :
- `text_nettoye` : le texte nettoyé
- `sentiment` : le label positif/negatif/neutre

In [7]:
# Garder uniquement les colonnes utiles
df_final = df[["text_nettoye", "sentiment"]].copy()

# Sauvegarder dans le dossier output/
df_final.to_csv("../outputs/02_data_nettoyee.csv", index=False)

# Vérifier la sauvegarde
df_verif = pd.read_csv("../outputs/02_data_nettoyee.csv")
print(f" Fichier sauvegardé avec succès !")
print(f"Lignes : {len(df_verif)}")
print(f"Colonnes : {df_verif.columns.tolist()}")
print(f"\nAperçu :")
df_verif.head(3)

 Fichier sauvegardé avec succès !
Lignes : 568446
Colonnes : ['text_nettoye', 'sentiment']

Aperçu :


,text_nettoye,sentiment
0,bought several vitality canned dog food produc...,positif
1,product arrived labeled jumbo salted peanutsth...,negatif
2,confection around century light pillowy citrus...,positif


##  Résumé de ce notebook

Dans ce notebook on a :

1. **Chargé** les données du notebook précédent (568 454 avis)
2. **Nettoyé** le texte en 4 étapes :
   - Mise en minuscules
   - Suppression de la ponctuation et des chiffres
   - Suppression des stopwords (mots inutiles)
   - Lemmatisation (ramener les mots à leur forme de base)
3. **Vérifié** la qualité du nettoyage (réduction de 49%)
4. **Supprimé** 1 texte vide détecté après nettoyage
5. **Sauvegardé** le résultat dans `output/02_data_nettoyee.csv`
